# Incident Response Runbook: Discovery - Process Discovery

**Tactic:** Discovery
**Technique:** Process Discovery (T1057)
**Severity:** MEDIUM

## Overview

This runbook provides step-by-step incident response procedures for Process Discovery activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for process discovery indicators
print(f"\n[QUERY] Searching Splunk for process discovery indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=4688)
OR (sourcetype=linux_secure "ps " OR "tasklist" OR "top" OR "htop")
OR (sourcetype=network "process" OR "task" OR "enum")
| eval command=coalesce(CommandLine, cmd, command)
| where match(command, "tasklist|ps |top|htop|Get-Process|Get-WmiObject.*Win32_Process|enum.*process")
| stats count by host, user, command, _time
| where count > 5
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential process discovery events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and suspicious activities
affected_systems = []
process_discovery_indicators = []
unique_users = set()

for event in splunk_results:
    system_info = {
        'hostname': event.get('host', 'unknown'),
        'user': event.get('user', 'unknown'),
        'command': event.get('command', ''),
        'event_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)
    unique_users.add(event.get('user', 'unknown'))

    # Extract indicators
    if event.get('command'):
        process_discovery_indicators.append({
            'type': 'process_discovery_command',
            'value': event.get('command'),
            'context': f"Executed by {event.get('user', 'unknown')} on {event.get('host', 'unknown')}",
            'host': event.get('host', 'unknown')
        })

# Query CrowdStrike for endpoint detection
print(f"\n[QUERY] Checking CrowdStrike for process discovery detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Process Discovery'",
        start_time="-24h"
    )
    cs_affected_hosts = []
    for detection in cs_detections:
        host_info = {
            'device_id': detection.get('device_id', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0)
        }
        cs_affected_hosts.append(host_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['hostname'] == host_info['hostname']), None)
        if existing_system:
            existing_system.update(host_info)
        else:
            affected_systems.append(host_info)

    print(f"   Found {len(cs_affected_hosts)} CrowdStrike detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_affected_hosts = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in process_discovery_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            indicator['threat_intel'] = misp_search[0] if misp_search else None
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Process Discovery Incident - {len(affected_systems)} systems',
        'description': f'Detected process discovery activities on {len(affected_systems)} systems',
        'severity': 'MEDIUM',
        'tactic': 'Discovery',
        'technique': 'Process Discovery (T1057)',
        'indicators': process_discovery_indicators,
        'affected_systems': affected_systems,
        'threat_intelligence': misp_results
    }

    incident_id = iris.create_case(incident_data)
    if incident_id:
        print(f"   Created IRIS case: {incident_id}")
    else:
        print(f"   Failed to create IRIS case")
        incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - Affected systems: {len(affected_systems)}")
print(f"  - Unique users: {len(unique_users)}")
print(f"  - Suspicious indicators: {len(process_discovery_indicators)}")
print(f"  - Threat intelligence hits: {len(misp_results)}")
print(f"  - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []
containment_time = datetime.now().isoformat()

# 1. Isolate affected systems via CrowdStrike
print(f"\n[CONTAINMENT] Isolating affected systems...")
isolated_hosts = []
for system in affected_systems:
    try:
        result = crowdstrike.isolate_host(system['device_id'])
        if result:
            isolated_hosts.append(system['hostname'])
            containment_actions.append({
                'action': 'host_isolation',
                'target': system['hostname'],
                'device_id': system['device_id'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Isolated host: {system['hostname']}")
        else:
            print(f"   Failed to isolate host: {system['hostname']}")
    except Exception as e:
        print(f"   Error isolating {system['hostname']}: {e}")

# 2. Block suspicious IPs/domains via Shuffle
print(f"\n[CONTAINMENT] Blocking suspicious IPs/domains...")
blocked_indicators = []
for indicator in process_discovery_indicators:
    try:
        # Extract IPs from commands or logs
        ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
        ips = re.findall(ip_pattern, str(indicator))
        for ip in ips:
            if ip not in ['127.0.0.1', '0.0.0.0']:  # Skip localhost
                result = shuffle.block_ip(ip, f"Process discovery incident {incident_id}")
                if result:
                    blocked_indicators.append({'type': 'ip', 'value': ip})
                    containment_actions.append({
                        'action': 'ip_block',
                        'target': ip,
                        'reason': f'Process discovery from {indicator.get("host", "unknown")}',
                        'status': 'success',
                        'timestamp': containment_time
                    })
                    print(f"   Blocked IP: {ip}")

        # Extract domains from commands
        domain_pattern = r'\b([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b'
        domains = re.findall(domain_pattern, str(indicator))
        for domain in domains:
            if not any(skip in domain.lower() for skip in ['localhost', 'localdomain']):
                result = shuffle.block_domain(domain, f"Process discovery incident {incident_id}")
                if result:
                    blocked_indicators.append({'type': 'domain', 'value': domain})
                    containment_actions.append({
                        'action': 'domain_block',
                        'target': domain,
                        'reason': f'Process discovery from {indicator.get("host", "unknown")}',
                        'status': 'success',
                        'timestamp': containment_time
                    })
                    print(f"   Blocked domain: {domain}")
    except Exception as e:
        print(f"   Error blocking indicators: {e}")

# 3. Disable compromised user accounts
print(f"\n[CONTAINMENT] Disabling compromised user accounts...")
disabled_accounts = []
for user in unique_users:
    try:
        # Use Shuffle to disable accounts (assuming AD integration)
        result = shuffle.disable_user_account(user, f"Process discovery incident {incident_id}")
        if result:
            disabled_accounts.append(user)
            containment_actions.append({
                'action': 'account_disable',
                'target': user,
                'reason': 'Compromised account - process discovery activities',
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Disabled account: {user}")
        else:
            print(f"   Failed to disable account: {user}")
    except Exception as e:
        print(f"   Error disabling account {user}: {e}")

# 4. Enable enhanced monitoring
print(f"\n[CONTAINMENT] Enabling enhanced monitoring...")
try:
    # Create Splunk alerts for enhanced monitoring
    alert_configs = [
        {
            'name': f'Enhanced Process Discovery Monitoring - {incident_id}',
            'search': 'index=* sourcetype=WinEventLog:Security EventCode=4688 | eval risk_score = case(match(CommandLine, "tasklist|ps|get-process|wmic.*process"), 8, match(CommandLine, "systeminfo|hostname|whoami"), 6, 1==1, 3) | where risk_score >= 6',
            'alert_type': 'real-time',
            'severity': 'medium'
        }
    ]

    for config in alert_configs:
        result = splunk.create_alert(config)
        if result:
            containment_actions.append({
                'action': 'enhanced_monitoring',
                'target': 'splunk_alert',
                'alert_name': config['name'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Created enhanced monitoring alert: {config['name']}")

    # Enable CrowdStrike enhanced detection
    for system in affected_systems:
        result = crowdstrike.enable_enhanced_detection(system['device_id'])
        if result:
            containment_actions.append({
                'action': 'enhanced_edr',
                'target': system['hostname'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Enabled enhanced EDR for: {system['hostname']}")

except Exception as e:
    print(f"   Error enabling enhanced monitoring: {e}")

# Update IRIS case with containment actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'containment_actions': containment_actions,
            'containment_time': containment_time,
            'isolated_hosts': isolated_hosts,
            'blocked_indicators': blocked_indicators,
            'disabled_accounts': disabled_accounts
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with containment details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} systems isolated")
print(f"  - {len(blocked_indicators)} IPs/domains blocked")
print(f"  - {len(disabled_accounts)} accounts disabled")
print(f"  - Enhanced monitoring enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []
eradication_time = datetime.now().isoformat()

# 1. Terminate suspicious processes via CrowdStrike
print(f"\n[ERADICATION] Terminating suspicious processes...")
terminated_processes = []
for system in affected_systems:
    try:
        # Get running processes from CrowdStrike
        processes = crowdstrike.get_processes(system['device_id'])
        if processes:
            suspicious_processes = []
            for proc in processes:
                proc_name = proc.get('name', '').lower()
                proc_cmd = proc.get('command_line', '').lower()

                # Check against known discovery tools
                if any(tool in proc_cmd for tool in [
                    'nmap', 'masscan', 'zmap', 'bloodhound', 'powerview',
                    'dsquery', 'adfind', 'ldapsearch', 'mimikatz', 'empire'
                ]) or any(pattern in proc_cmd for pattern in [
                    'tasklist', 'ps ', 'get-process', 'wmic process',
                    'systeminfo', 'hostname', 'whoami', 'ipconfig',
                    'netstat', 'arp ', 'route print', 'net view'
                ]):
                    suspicious_processes.append(proc)

            # Terminate suspicious processes
            for proc in suspicious_processes[:5]:  # Limit to top 5 per host
                result = crowdstrike.terminate_process(system['device_id'], proc['pid'])
                if result:
                    terminated_processes.append({
                        'host': system['hostname'],
                        'process': proc['name'],
                        'pid': proc['pid'],
                        'command': proc.get('command_line', '')
                    })
                    eradication_actions.append({
                        'action': 'process_termination',
                        'target': system['hostname'],
                        'process': proc['name'],
                        'pid': proc['pid'],
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Terminated process: {proc['name']} (PID: {proc['pid']}) on {system['hostname']}")
    except Exception as e:
        print(f"   Error terminating processes on {system['hostname']}: {e}")

# 2. Delete discovery tools
print(f"\n[ERADICATION] Deleting discovery tools...")
deleted_tools = []
for indicator in process_discovery_indicators:
    try:
        if indicator.get('command'):
            # Extract file paths from commands
            file_paths = re.findall(r'([C-Z]:[^\s"]*\.(exe|bat|ps1|vbs|py))', indicator['command'], re.IGNORECASE)
            for file_path, ext in file_paths:
                result = crowdstrike.delete_file(indicator['host'], file_path)
                if result:
                    deleted_tools.append({
                        'host': indicator['host'],
                        'file': file_path
                    })
                    eradication_actions.append({
                        'action': 'tool_deletion',
                        'target': indicator['host'],
                        'file': file_path,
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Deleted tool: {file_path} on {indicator['host']}")

        # Clean up temp directories
        temp_dirs = [
            f"C:\\Users\\{indicator.get('user', 'unknown')}\\AppData\\Local\\Temp\\*",
            f"C:\\Users\\{indicator.get('user', 'unknown')}\\Downloads\\*",
            "C:\\Windows\\Temp\\*"
        ]
        for temp_dir in temp_dirs:
            result = crowdstrike.delete_file(indicator['host'], temp_dir)
            if result:
                print(f"   Cleaned temp directory: {temp_dir} on {indicator['host']}")
    except Exception as e:
        print(f"   Error deleting tools: {e}")

# 3. Reset user credentials
print(f"\n[ERADICATION] Resetting compromised user credentials...")
reset_credentials = []
for user in unique_users:
    try:
        # Reset password
        result = shuffle.reset_user_password(user, f"Process discovery compromise - incident {incident_id}")
        if result:
            reset_credentials.append(user)
            eradication_actions.append({
                'action': 'credential_reset',
                'target': user,
                'reason': 'Process discovery activities detected',
                'status': 'success',
                'timestamp': eradication_time
            })
            print(f"   Reset credentials for: {user}")
        else:
            print(f"   Failed to reset credentials for: {user}")
    except Exception as e:
        print(f"   Error resetting credentials for {user}: {e}")

# 4. Clean up scheduled tasks or persistence
print(f"\n[ERADICATION] Cleaning up persistence mechanisms...")
cleaned_persistence = []
for system in affected_systems:
    try:
        # Check for scheduled tasks created for discovery
        scheduled_tasks = crowdstrike.get_scheduled_tasks(system['device_id'])
        if scheduled_tasks:
            for task in scheduled_tasks:
                task_name = task.get('name', '').lower()
                task_cmd = task.get('command', '').lower()

                if any(suspicious in task_cmd for suspicious in [
                    'nmap', 'scan', 'recon', 'discover', 'enum',
                    'tasklist', 'systeminfo', 'netstat'
                ]):
                    result = crowdstrike.delete_scheduled_task(system['device_id'], task['name'])
                    if result:
                        cleaned_persistence.append({
                            'host': system['hostname'],
                            'task': task['name']
                        })
                        eradication_actions.append({
                            'action': 'scheduled_task_cleanup',
                            'target': system['hostname'],
                            'task': task['name'],
                            'status': 'success',
                            'timestamp': eradication_time
                        })
                        print(f"   Removed suspicious scheduled task: {task['name']} on {system['hostname']}")
    except Exception as e:
        print(f"   Error cleaning persistence on {system['hostname']}: {e}")

# 5. Verify eradication
print(f"\n[ERADICATION] Verifying eradication...")
verification_results = []
try:
    # Re-scan systems for remaining discovery tools
    for system in affected_systems:
        scan_result = crowdstrike.scan_for_discovery_tools(system['device_id'])
        if scan_result:
            verification_results.append({
                'host': system['hostname'],
                'scan_result': scan_result,
                'threats_found': len(scan_result.get('discovery_threats', []))
            })
            if scan_result.get('discovery_threats'):
                print(f"   Verification found {len(scan_result['discovery_threats'])} remaining discovery tools on {system['hostname']}")
            else:
                print(f"   Verification clean: {system['hostname']}")
        else:
            print(f"   Verification scan failed for: {system['hostname']}")
except Exception as e:
    print(f"   Error during verification: {e}")

# Update IRIS case with eradication actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'eradication_actions': eradication_actions,
            'eradication_time': eradication_time,
            'terminated_processes': terminated_processes,
            'deleted_tools': deleted_tools,
            'reset_credentials': reset_credentials,
            'cleaned_persistence': cleaned_persistence,
            'verification_results': verification_results
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with eradication details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(terminated_processes)} suspicious processes terminated")
print(f"  - {len(deleted_tools)} discovery tools deleted")
print(f"  - {len(reset_credentials)} user credentials reset")
print(f"  - {len(cleaned_persistence)} persistence mechanisms cleaned")
print(f"  - Verification completed for {len(verification_results)} systems")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []
recovery_time = datetime.now().isoformat()

# 1. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
reenabled_hosts = []
for system in affected_systems:
    if system['hostname'] in isolated_hosts:
        try:
            result = crowdstrike.unisolate_host(system['device_id'])
            if result:
                reenabled_hosts.append(system['hostname'])
                recovery_actions.append({
                    'action': 'host_unisolation',
                    'target': system['hostname'],
                    'device_id': system['device_id'],
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Re-enabled host: {system['hostname']}")
            else:
                print(f"   Failed to re-enable host: {system['hostname']}")
        except Exception as e:
            print(f"   Error re-enabling {system['hostname']}: {e}")

# 2. Restore normal user access
print(f"\n[RECOVERY] Restoring normal user access...")
restored_access = []
for user in disabled_accounts:
    try:
        # Re-enable user account
        result = shuffle.enable_user_account(user, f"Recovery complete - process discovery incident {incident_id}")
        if result:
            restored_access.append(user)
            recovery_actions.append({
                'action': 'account_enable',
                'target': user,
                'reason': 'Recovery complete - process discovery incident resolved',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored access for: {user}")
        else:
            print(f"   Failed to restore access for: {user}")
    except Exception as e:
        print(f"   Error restoring access for {user}: {e}")

# 3. Validate system functionality
print(f"\n[RECOVERY] Validating system functionality...")
validation_results = []
for system in affected_systems:
    try:
        # Check system health via CrowdStrike
        health_check = crowdstrike.check_system_health(system['device_id'])
        if health_check:
            validation_results.append({
                'host': system['hostname'],
                'health_status': health_check.get('status', 'unknown'),
                'processes_normal': health_check.get('process_status', 'unknown'),
                'connectivity': health_check.get('network_status', 'unknown')
            })

            if health_check.get('status') == 'healthy':
                recovery_actions.append({
                    'action': 'health_validation',
                    'target': system['hostname'],
                    'status': 'success',
                    'health_status': 'healthy',
                    'timestamp': recovery_time
                })
                print(f"   System health validated: {system['hostname']}")
            else:
                print(f"   System health issues detected: {system['hostname']} - {health_check.get('issues', [])}")
        else:
            print(f"   Health check failed for: {system['hostname']}")
    except Exception as e:
        print(f"   Error validating {system['hostname']}: {e}")

# 4. Restore normal monitoring levels
print(f"\n[RECOVERY] Restoring normal monitoring levels...")
try:
    # Disable enhanced process discovery monitoring
    result = splunk.disable_alert(f'Enhanced Process Discovery Monitoring - {incident_id}')
    if result:
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'splunk_alerts',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring levels")

    # Restore normal CrowdStrike monitoring
    for system in affected_systems:
        result = crowdstrike.restore_normal_monitoring(system['device_id'])
        if result:
            recovery_actions.append({
                'action': 'edr_monitoring_restore',
                'target': system['hostname'],
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored normal EDR monitoring for: {system['hostname']}")

except Exception as e:
    print(f"   Error restoring monitoring: {e}")

# 5. Monitor for residual discovery attempts
print(f"\n[RECOVERY] Establishing post-recovery monitoring...")
try:
    # Create follow-up monitoring for process discovery
    followup_alerts = [
        {
            'name': f'Post-Recovery Process Discovery Monitoring - {incident_id}',
            'search': f'index=* host IN ({",".join([f'"{s["hostname"]}"' for s in affected_systems])}) sourcetype=WinEventLog:Security EventCode=4688 | eval risk_score = case(match(CommandLine, "tasklist|ps|get-process|systeminfo"), 7, match(CommandLine, "nmap|dsquery|bloodhound"), 9, 1==1, 3) | where risk_score >= 7',
            'alert_type': 'scheduled',
            'severity': 'medium',
            'schedule': '0 */8 * * *'  # Every 8 hours
        }
    ]

    for alert_config in followup_alerts:
        result = splunk.create_alert(alert_config)
        if result:
            recovery_actions.append({
                'action': 'followup_monitoring',
                'target': 'splunk_alert',
                'alert_name': alert_config['name'],
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Created follow-up monitoring: {alert_config['name']}")

except Exception as e:
    print(f"   Error establishing follow-up monitoring: {e}")

# Update IRIS case with recovery actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'recovery_actions': recovery_actions,
            'recovery_time': recovery_time,
            'reenabled_hosts': reenabled_hosts,
            'restored_access': restored_access,
            'validation_results': validation_results
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with recovery details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(reenabled_hosts)} systems re-enabled")
print(f"  - {len(restored_access)} user accounts restored")
print(f"  - System validation completed for {len(validation_results)} systems")
print(f"  - Normal monitoring levels restored")
print(f"  - Post-recovery monitoring established")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': f'Process Discovery Incident Report - {len(process_discovery_indicators)} indicators',
        'detection_time': detection_time,
        'closure_time': closure_time,
        'severity': 'MEDIUM',
        'tactic': 'Discovery',
        'technique': 'Process Discovery (T1057)',
        'summary': {
            'affected_users': len(unique_users),
            'affected_systems': len(affected_systems),
            'indicators_detected': len(process_discovery_indicators),
            'hosts_isolated': len(isolated_hosts),
            'accounts_disabled': len(disabled_accounts),
            'processes_terminated': len(terminated_processes),
            'tools_deleted': len(deleted_tools),
            'credentials_reset': len(reset_credentials)
        },
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': closure_time
        },
        'actions_taken': {
            'detection': process_discovery_indicators[:10],  # Top 10 indicators
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'threat_intelligence': [i.get('threat_intel') for i in process_discovery_indicators if i.get('threat_intel')],
        'recommendations': [
            'Implement process discovery monitoring and alerting',
            'Restrict access to system enumeration commands',
            'Deploy endpoint detection for reconnaissance tools',
            'Regular security awareness training on reconnaissance detection',
            'Enhance logging for process and system discovery activities'
        ]
    }

    # Save report to file
    report_filename = f"process_discovery_report_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'report_generation',
        'target': report_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Generated incident report: {report_filename}")

except Exception as e:
    print(f"   Error generating report: {e}")

# 2. Document lessons learned
print(f"\n[POST-INCIDENT] Documenting lessons learned...")
try:
    lessons_learned = {
        'incident_id': incident_id,
        'what_went_well': [
            'Rapid detection of process discovery activities',
            'Effective isolation of affected systems',
            'Comprehensive eradication of discovery tools',
            'Successful system recovery and validation'
        ],
        'what_could_improve': [
            'Earlier detection of reconnaissance patterns',
            'Enhanced monitoring of system enumeration',
            'Automated blocking of discovery commands',
            'Better user training on suspicious activity reporting'
        ],
        'key_findings': [
            f'Process discovery affected {len(unique_users)} users across {len(affected_systems)} systems',
            f'Most common discovery method: {max([i.get("type", "unknown") for i in process_discovery_indicators], key=[i.get("type", "unknown") for i in process_discovery_indicators].count)}',
            'Threat intelligence enriched detection for key reconnaissance tools',
            'Automated response prevented further lateral movement'
        ],
        'preventive_measures': [
            'Implement command-line auditing and restrictions',
            'Deploy advanced endpoint protection with behavior monitoring',
            'Regular vulnerability scanning and system hardening',
            'Enhanced security monitoring and alerting',
            'Security awareness training focused on reconnaissance detection'
        ]
    }

    # Save lessons learned
    lessons_filename = f"process_discovery_lessons_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(lessons_filename, 'w') as f:
        json.dump(lessons_learned, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'lessons_learned',
        'target': lessons_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Documented lessons learned: {lessons_filename}")

except Exception as e:
    print(f"   Error documenting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_measures = []

    # Update Splunk correlation rules for process discovery
    updated_rules = splunk.update_correlation_rules([
        {
            'name': 'Enhanced Process Discovery Detection',
            'search': 'index=* sourcetype=WinEventLog:Security EventCode=4688 | eval risk_score = case(match(CommandLine, "tasklist|ps|get-process|wmic.*process"), 8, match(CommandLine, "systeminfo|hostname|whoami|netstat"), 7, match(CommandLine, "nmap|dsquery|bloodhound"), 9, 1==1, 4) | where risk_score >= 7',
            'alert_threshold': 5,
            'time_window': '15m'
        }
    ])
    if updated_rules:
        preventive_measures.append('Updated Splunk process discovery rules')
        print(f"   Updated Splunk correlation rules for process discovery detection")

    # Enhance CrowdStrike prevention policies
    cs_policies = crowdstrike.update_prevention_policies({
        'process_discovery_prevention': 'enabled',
        'command_line_auditing': 'strict',
        'reconnaissance_tool_detection': 'enabled',
        'system_enumeration_monitoring': 'enabled'
    })
    if cs_policies:
        preventive_measures.append('Enhanced CrowdStrike discovery policies')
        print(f"   Enhanced CrowdStrike prevention policies")

    # Schedule process discovery security training
    training_scheduled = shuffle.schedule_security_training(
        users=list(unique_users),
        topic='Detecting and Reporting Reconnaissance Activities',
        incident_reference=incident_id
    )
    if training_scheduled:
        preventive_measures.append('Scheduled reconnaissance training')
        print(f"   Scheduled security awareness training on reconnaissance detection")

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Share threat intelligence
print(f"\n[POST-INCIDENT] Sharing threat intelligence...")
try:
    shared_intel = []
    for indicator in process_discovery_indicators:
        if indicator.get('threat_intel'):
            # Share with MISP
            result = misp.share_indicator(indicator, incident_id)
            if result:
                shared_intel.append(f"MISP: {indicator.get('type', 'unknown')}")
                print(f"   Shared indicator with MISP: {indicator.get('type', 'unknown')}")

    if shared_intel:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': shared_intel,
            'status': 'success',
            'timestamp': closure_time
        })

except Exception as e:
    print(f"   Error sharing threat intelligence: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    if incident_id:
        closure_data = {
            'status': 'closed',
            'closure_time': closure_time,
            'closure_reason': 'Process discovery incident successfully contained, eradicated, and recovered',
            'post_incident_actions': post_incident_actions,
            'final_assessment': {
                'threat_contained': len(isolated_hosts) > 0,
                'threat_eradicated': len(terminated_processes) > 0 or len(deleted_tools) > 0,
                'systems_recovered': len(reenabled_hosts) > 0,
                'preventive_measures': len(preventive_measures) > 0
            }
        }

        result = iris.close_case(incident_id, closure_data)
        if result:
            post_incident_actions.append({
                'action': 'case_closure',
                'target': incident_id,
                'status': 'success',
                'timestamp': closure_time
            })
            print(f"   Closed IRIS case: {incident_id}")
        else:
            print(f"   Failed to close IRIS case: {incident_id}")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated")
print(f"  - Lessons learned documented")
print(f"  - {len(preventive_measures)} preventive measures implemented")
print(f"  - Threat intelligence shared with {len(shared_intel)} platforms")
print(f"  - Incident case closed: {incident_id}")

print(f"\n Process Discovery Incident Response Complete")
print(f"   Total duration: {(datetime.fromisoformat(closure_time) - datetime.fromisoformat(detection_time)).total_seconds() / 3600:.1f} hours")
print(f"   Response effectiveness: {'HIGH' if len(isolated_hosts) > 0 and len(terminated_processes) > 0 else 'MEDIUM'}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
